# Phase 3 — Feature Engineering & Preprocessing
**Capstone 1 | Phases 1–5**

Builds the full 800-feature matrix:
- 14 intrinsic stylometric features
- 16 advanced linguistic features (TextBlob sentiment, stopword ratio, punctuation counts, etc.)
- TF-IDF word n-grams (up to 2,000 features, ngram_range=(1,3))
- Character n-grams (up to 500 features, ngram_range=(2,4))
- SelectKBest(chi2, k=800) → StandardScaler → 80/20 stratified split

**Loads:** `outputs/df_cleaned.pkl`  
**Saves:** `outputs/features_baseline.pkl`  
**Next:** `phase4_baseline_models.ipynb`


In [ ]:
import numpy as np, random, os
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    print(f"Seeds fixed (numpy + random + torch): {RANDOM_SEED}")
except ImportError:
    print(f"Seeds fixed (numpy + random): {RANDOM_SEED}")
print("Always run this cell first on every restart.")


In [ ]:
import joblib
df = joblib.load('outputs/df_cleaned.pkl')
print(f"Loaded df: {df.shape}")


In [ ]:
# advanced feature extraction
try:
    stopwords = stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stopwords = stopwords.words('english')

try:
    word_tokenize('test')
except LookupError:
    nltk.download('punkt')

try:
    sent_tokenize('test')
except LookupError:
    nltk.download('punkt_tab')

df_features = df.copy()

def extract_advanced_features(text):
    """
    Extract comprehensive linguistic and stylometric features.

    These features are crucial for AI vs. Human detection because they capture
    subtle statistical differences (like consistency in word choice, sentence structure,
    and punctuation usage) that often appear in synthetic data.
    """
    if pd.isna(text) or text == '':
        return {
            'sentiment_polarity': 0,
            'sentiment_subjectivity': 0,
            'num_sentences': 0,
            'avg_word_length': 0,
            'unique_word_ratio': 0,
            'stopword_ratio': 0,
            'uppercase_ratio': 0,
            'digit_ratio': 0,
            'special_char_ratio': 0,
            'avg_sentence_length': 0,
            'long_word_ratio': 0,
            'question_count': 0,
            'exclamation_count': 0,
            'comma_ratio': 0,
            'semicolon_ratio': 0,
            'word_diversity': 0
        }

    # Ensure text is a string before processing
    text = str(text)

    # TextBlob analysis
    blob = TextBlob(text)

    # Basic Tokenization and Counting
    words = word_tokenize(text)
    num_words = len(words)
    unique_words = len(set(words))

    #  Stopword Ratio (Functional Vocabulary)
    # Measures the proportion of common, functional words (e.g., 'the', 'is', 'a').
    # AI models may exhibit unnatural stopword frequencies compared to human baselines.
    stop_words = set(stopwords)
    stopword_count = sum(1 for word in words if word.lower() in stop_words)

    # Character-Level Analysis (Low-Level Fingerprints)
    # These low-level features can catch unusual capitalization patterns or non-text content.
    num_uppercase = sum(1 for c in text if c.isupper())
    num_digits = sum(1 for c in text if c.isdigit())
    num_special = sum(1 for c in text if not c.isalnum() and not c.isspace())

    # Sentence analysis
    sentences = sent_tokenize(text)
    num_sentences = len(sentences)

    # Word Length and Complexity
    # Measures how complex the vocabulary is. AI models sometimes converge on specific word lengths.
    word_lengths = [len(word) for word in words]
    long_words = sum(1 for length in word_lengths if length > 6)

    # Punctuation Analysis
    # Punctuation usage (especially commas and semicolons) is difficult
    # for AI to perfectly replicate, making these effective stylometric features.
    comma_count = text.count(',')
    semicolon_count = text.count(';')

    return {
        # Sentiment metrics: AI text might cluster around neutral polarity and low subjectivity.
        'sentiment_polarity': blob.sentiment.polarity,
        'sentiment_subjectivity': blob.sentiment.subjectivity,
        # Basic Structure Metrics
        'num_sentences': num_sentences,
        'avg_word_length': np.mean(word_lengths) if word_lengths else 0,
        # Lexical Richness and Efficiency
        # Unique Word Ratio: Simple measure of lexical diversity.
        'unique_word_ratio': unique_words / num_words if num_words > 0 else 0,
        # Stopword Ratio: Deviation from human norm suggests a synthetic pattern.
        'stopword_ratio': stopword_count / num_words if num_words > 0 else 0,
        # Low-Level Character Patterns
        'uppercase_ratio': num_uppercase / len(text) if len(text) > 0 else 0,
        'digit_ratio': num_digits / len(text) if len(text) > 0 else 0,
        'special_char_ratio': num_special / len(text) if len(text) > 0 else 0,
        # Sentence/Structural Flow
        # Avg Sentence Length: A measure of stylistic complexity.
        'avg_sentence_length': num_words / num_sentences if num_sentences > 0 else 0,
        # Long Word Ratio: Another measure of vocabulary complexity.
        'long_word_ratio': long_words / num_words if num_words > 0 else 0,
        'question_count': text.count('?'),
        'exclamation_count': text.count('!'),
        # Punctuation ratios based on word count for normalization.
        'comma_ratio': comma_count / num_words if num_words > 0 else 0,
        'semicolon_ratio': semicolon_count / num_words if num_words > 0 else 0,
        # Word Diversity: Yule's K or Guiraud's root TTR (unique_words / sqrt(num_words)).
        # A more stable measure of lexical richness than simple TTR (unique_word_ratio).
        'word_diversity': unique_words / np.sqrt(num_words) if num_words > 0 else 0
    }

# Extract features
print("Extracting advanced text features...")
# Apply the feature extraction function to the text column of the DataFrame.
text_features = df_features['text_content'].apply(extract_advanced_features)
# Convert the list of feature dictionaries into a new DataFrame.
text_features_df = pd.DataFrame(list(text_features))

# Display the first few rows of the new features dataframe
display(text_features_df.head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Extracting advanced text features...


,sentiment_polarity,sentiment_subjectivity,num_sentences,avg_word_length,unique_word_ratio,stopword_ratio,uppercase_ratio,digit_ratio,special_char_ratio,avg_sentence_length,long_word_ratio,question_count,exclamation_count,comma_ratio,semicolon_ratio,word_diversity
0,0.128860,0.400194,54,4.795322,0.774854,0.093567,0.029061,0.0,0.028023,6.333333,0.233918,0,0,0.0,0.0,14.329559
1,0.061689,0.343355,44,4.939394,0.804714,0.070707,0.027341,0.0,0.026178,6.750000,0.262626,0,0,0.0,0.0,13.868187
2,0.070141,0.439260,75,4.909091,0.727273,0.096970,0.029835,0.0,0.026325,6.600000,0.284848,0,0,0.0,0.0,16.180797
3,0.012469,0.381643,34,4.847826,0.808696,0.073913,0.026718,0.0,0.025954,6.764706,0.226087,0,0,0.0,0.0,12.264477
4,0.164394,0.402453,28,5.085106,0.808511,0.053191,0.026009,0.0,0.025112,6.714286,0.281915,0,0,0.0,0.0,11.085739


In [ ]:
# Comprehensive Preprocessing and Feature Engineering

# Separate target
y_full = df_features['label']

# Get numeric features (excluding original text_content and content_type)
X_numeric_raw = df_features.drop(['label', 'text_content', 'content_type'], axis=1)

# Handle missing values in numeric features
# This step ensures robustness, as ML models cannot process NaN values.
numerical_cols_raw = X_numeric_raw.select_dtypes(include=['int64', 'float64']).columns
X_numeric_raw[numerical_cols_raw] = X_numeric_raw[numerical_cols_raw].fillna(X_numeric_raw[numerical_cols_raw].median())


# One-hot encode content_type
content_type_encoded = pd.get_dummies(df_features['content_type'], prefix='content', drop_first=True)

# Combine numeric features
X_numeric_combined = pd.concat([X_numeric_raw, text_features_df, content_type_encoded], axis=1)

print(f"Numeric + Engineered features: {X_numeric_combined.shape[1]}")
print("\nExtracting TF-IDF features...")


# TF-IDF (Term Frequency-Inverse Document Frequency) quantifies the importance of a word
# by weighting its frequency in a document against its inverse frequency across all documents.
# This captures *semantic* content differences between AI and human text.
tfidf = TfidfVectorizer(
    max_features=2000, # Limits the vocabulary to the 2000 most discriminative words to manage high dimensionality and noise.
    ngram_range=(1, 3),
    min_df=3, # Ignores terms that appear in fewer than 3 documents (removes noise from very rare words/typos).
    max_df=0.85, # Ignores terms in more than 85% of documents (removes overly common, non-discriminative words).
    strip_accents='unicode',
    strip_accents='unicode',
    lowercase=True,
    sublinear_tf=True, # Applies the sublinear TF scaling (1 + log(tf)) to reduce the impact of very high-frequency words.
    stop_words='english' # Removes common English function words to focus on more meaningful terms.
)

# Fit the vectorizer to the text and transform the data.
X_tfidf = tfidf.fit_transform(df_features['text_content']).toarray()
print(f"TF-IDF features: {X_tfidf.shape[1]}")

# Character n-grams are excellent for **stylometry** (authorship detection). They capture sub-word
# patterns (like punctuation, common prefixes/suffixes) that define writing style, often different
# between human variability and AI consistency.
char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 4),
    max_features=500
)
X_char = char_vectorizer.fit_transform(df_features['text_content']).toarray()
print(f"Character n-gram features: {X_char.shape[1]}")
print("\nApplying feature selection...")

# Combining all features
X_all_features = np.hstack([
    X_numeric_combined.values,
    X_tfidf,
    X_char
])

print(f"Total features before selection: {X_all_features.shape[1]}")

# Select top features using chi2
# Using absolute values for chi2 as it doesn't work with negative values
selector = SelectKBest(chi2, k=min(800, X_all_features.shape[1]))
X_selected = selector.fit_transform(np.abs(X_all_features), y_full)
print(f"Features after selection: {X_selected.shape[1]}")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_full,
    test_size=0.2,  # 20% for testing
    random_state=42, # for reproducibility
    stratify=y_full # to maintain class distribution
)

print("\nData splitting complete.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Numeric + Engineered features: 37

Extracting TF-IDF features...
TF-IDF features: 2000
Character n-gram features: 500

Applying feature selection...
Total features before selection: 2537
Features after selection: 800

Data splitting complete.
X_train shape: (1093, 800)
X_test shape: (274, 800)
y_train shape: (1093,)
y_test shape: (274,)


### Summary:

Feature Engineering and Preprocessing:

Extracted a rich set of features from the text content:

* Intrinsic Features: Basic text metrics like word_count, character_count, sentence_count, lexical_diversity, avg_sentence_length, avg_word_length, and punctuation_ratio were directly used or calculated.

* Advanced Linguistic Features: Custom functions were used to derive features such as sentiment_polarity, sentiment_subjectivity, unique_word_ratio, stopword_ratio, uppercase_ratio, digit_ratio, special_char_ratio, long_word_ratio, various punctuation counts (question_count, exclamation_count, comma_ratio, semicolon_ratio), and word_diversity.

* Categorical Encoding: The content_type column was one-hot encoded.

* TF-IDF Features: Generated TF-IDF vectors for word n-grams (max 2000 features, range 1-3) to capture term importance.

* Character N-gram Features: TF-IDF was also applied to character n-grams (max 500 features, range 2-4) to capture stylistic elements.

* Feature Selection: All combined features were subjected to SelectKBest with chi2 to select the top 800 most informative features, reducing dimensionality and noise.

* Feature Scaling: The selected features were then scaled using StandardScaler to ensure all features contributed equally to model training.

* Data Splitting: The processed data was split into training and testing sets (80/20 ratio) using stratify to maintain class distribution.


In [ ]:
import joblib
joblib.dump({
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
    'tfidf': tfidf, 'char': char_vectorizer,
    'selector': selector, 'scaler': scaler,
    'df': df,
}, 'outputs/features_baseline.pkl')
print(f"Saved outputs/features_baseline.pkl")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
